# Đo tỉ lệ dòng bị loại theo từng luật chất lượng, NYC TLC 2019-01

Mẫu 200 000 chuyến (`seed=0`). Điền 4 hàm trong các cell chấm điểm; cell `explore` chỉ để xem.

In [ ]:
import pandas as pd

MONTH = "2019-01"
SAMPLE_N = 200_000
SEED = 0
COLUMNS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "PULocationID", "DOLocationID"]
RULES = ["duration", "distance", "pickup_month", "zone_range", "zone_unknown"]

In [ ]:
def load_sample(path, n=SAMPLE_N, seed=SEED):
    df = pd.read_parquet(path, columns=COLUMNS)
    return df.sample(n=min(n, len(df)), random_state=seed)

In [ ]:
def rule_masks(df, month):
    start = pd.Timestamp(month + "-01")
    end = start + pd.offsets.MonthBegin(1)
    pu, do = df["PULocationID"], df["DOLocationID"]
    return {
        "duration": df["tpep_dropoff_datetime"] <= df["tpep_pickup_datetime"],
        "distance": df["trip_distance"] <= 0,
        "pickup_month": (df["tpep_pickup_datetime"] < start) | (df["tpep_pickup_datetime"] >= end),
        "zone_range": ~pu.between(1, 265) | ~do.between(1, 265),
        "zone_unknown": pu.isin([264, 265]) | do.isin([264, 265]),
    }

In [ ]:
def rejection_counts(df, month):
    return {name: int(mask.sum()) for name, mask in rule_masks(df, month).items()}

In [ ]:
def rows_kept(df, month):
    bad = pd.concat(rule_masks(df, month).values(), axis=1).any(axis=1)
    return int((~bad).sum())

In [ ]:
# explore: đọc tháng thật và xem tỉ lệ từng luật
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sample = load_sample(ROOT / "data" / "raw" / "nyc-tlc" / "yellow_tripdata_2019-01.parquet")
counts = rejection_counts(sample, MONTH)
rates = pd.Series(counts) / len(sample) * 100
print(rates.round(3))
print("giữ lại:", rows_kept(sample, MONTH), "trên", len(sample))

In [ ]:
# explore: biểu đồ tỉ lệ loại theo luật
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ax = rates.sort_values().plot.barh()
ax.set_xlabel("% dòng vi phạm trên mẫu 200 000")
plt.tight_layout()
plt.savefig("rejection_by_rule.png")
print("pickup nhỏ nhất/lớn nhất:", sample["tpep_pickup_datetime"].min(), sample["tpep_pickup_datetime"].max())

Làm phần trắc nghiệm trong app sau khi chạy notebook.